# 01 — EDA & Preprocessing
Credit Score Classification Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

SEED = 42
print('Ready.')

In [ ]:
df = pd.read_csv('data/train.csv', low_memory=False)
print(f'Shape: {df.shape} | Unique customers: {df["Customer_ID"].nunique()}')
print(f'\nTarget distribution:\n{df["Credit_Score"].value_counts()}')

In [ ]:
# Class imbalance plot
order = ['Poor', 'Standard', 'Good']
colors = ['#E8A838', '#E8622A', '#D84B6E']
plt.figure(figsize=(7,5))
df['Credit_Score'].value_counts()[order].plot(kind='bar', color=colors, edgecolor='white')
plt.title('Class Distribution of Credit Scores', fontsize=13)
plt.xlabel('Credit Score Category'); plt.ylabel('Number of Records')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150); plt.show()

In [ ]:
# Correlation heatmap on key numeric features
numeric_sample = ['Annual_Income', 'Outstanding_Debt', 'Num_of_Loan',
                   'Credit_Utilization_Ratio', 'Delay_from_due_date', 'Total_EMI_per_month']
df_num = df[numeric_sample].apply(pd.to_numeric, errors='coerce').dropna()
plt.figure(figsize=(8,6))
sns.heatmap(df_num.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=['Income','Debt','Loans','Utilization','Delay','EMI'],
            yticklabels=['Income','Debt','Loans','Utilization','Delay','EMI'])
plt.title('Correlation Matrix'); plt.tight_layout()
plt.savefig('results/correlation_matrix.png', dpi=150); plt.show()

In [ ]:
# Full cleaning pipeline (see combined notebook for detailed comments)
df_clean = df.copy()
drop_cols = ['Name', 'SSN', 'ID']
df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns], inplace=True)

numeric_cols = ['Age','Annual_Income','Monthly_Inhand_Salary','Num_Bank_Accounts',
                'Num_Credit_Card','Interest_Rate','Num_of_Loan','Delay_from_due_date',
                'Num_of_Delayed_Payment','Changed_Credit_Limit','Num_Credit_Inquiries',
                'Outstanding_Debt','Credit_Utilization_Ratio','Total_EMI_per_month',
                'Amount_invested_monthly','Monthly_Balance']
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col].astype(str).str.replace('[^0-9.-]','',regex=True), errors='coerce')

df_clean['Monthly_Inhand_Salary'] = df_clean.groupby('Customer_ID')['Monthly_Inhand_Salary'].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill'))
if 'Credit_Mix' in df_clean.columns:
    df_clean['Credit_Mix'].fillna(df_clean['Credit_Mix'].mode()[0], inplace=True)
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)
for col in numeric_cols:
    if col in df_clean.columns:
        p05, p95 = df_clean[col].quantile(0.05), df_clean[col].quantile(0.95)
        df_clean[col] = df_clean[col].clip(lower=p05, upper=p95)

df_clean['Debt_Income_Ratio'] = df_clean['Outstanding_Debt'] / (df_clean['Annual_Income'] + 1)
if 'Payment_Behaviour' in df_clean.columns:
    spending_map = {'Low_spent':0,'Medium_spent':1,'High_spent':2}
    payment_map = {'Small_value':0,'Medium_value':1,'Large_value':2}
    df_clean['Spending_Level_Num'] = df_clean['Payment_Behaviour'].str.extract(r'(Low_spent|Medium_spent|High_spent)')[0].map(spending_map).fillna(1)
    df_clean['Payment_Value_Num'] = df_clean['Payment_Behaviour'].str.extract(r'(Small_value|Medium_value|Large_value)')[0].map(payment_map).fillna(1)
    df_clean.drop(columns=['Payment_Behaviour'], inplace=True)
if 'Type_of_Loan' in df_clean.columns:
    df_clean['Num_Loan_Types'] = df_clean['Type_of_Loan'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) and x != 'Not Specified' else 0)
    df_clean.drop(columns=['Type_of_Loan'], inplace=True)
month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
if 'Month' in df_clean.columns:
    df_clean['Month_ordinal'] = df_clean['Month'].map(month_map).fillna(0).astype(int)
    df_clean.drop(columns=['Month'], inplace=True)
if 'Occupation' in df_clean.columns:
    df_clean = pd.get_dummies(df_clean, columns=['Occupation'], drop_first=True)
if 'Credit_Mix' in df_clean.columns:
    df_clean['Credit_Mix'] = df_clean['Credit_Mix'].map({'Bad':0,'Standard':1,'Good':2}).fillna(1)
target_map = {'Poor':0,'Standard':1,'Good':2}
df_clean['Credit_Score_Enc'] = df_clean['Credit_Score'].map(target_map)
df_clean.drop(columns=['Credit_Score'], inplace=True)

df_clean.to_csv('data/cleaned.csv', index=False)
print(f'Saved cleaned data: {df_clean.shape}')